# 02. Geospatial Analysis & Unsupervised Hotspot Detection
### Project: Smart Public Safety Analytics
**Focus:** Geographic coordinate validation, density mapping, DBSCAN clustering (haversine metric), and K-Means comparison.

> **CRITICAL METHODOLOGICAL NOTICE:**
> Detected clusters represent **historical incident concentrations in reported public data**. They must never be interpreted as predictive indicators of future crime.


In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import pandas as pd
import numpy as np
import folium

import config
from src.data_loader import load_sample_dataset
from src.data_cleaning import clean_and_validate_data
from src.feature_engineering import engineer_features
from src.spatial_analysis import get_spatial_summary, get_area_summary
from src.hotspot_detection import run_dbscan_clustering, run_kmeans_comparison


## 1. Load and Clean Spatial Incident Data


In [ ]:
raw_df, mapping, _, _ = load_sample_dataset()
cleaned_df, _ = clean_and_validate_data(raw_df, mapping)
featured_df = engineer_features(cleaned_df)

spatial_summary = get_spatial_summary(featured_df)
print("Spatial Bounding Box Summary:")
for k, v in spatial_summary.items():
    print(f" - {k}: {v}")


## 2. Unsupervised Hotspot Detection with DBSCAN
Using haversine spherical distance metric on geographic coordinates in radians:
$$\epsilon_{\text{rad}} = \frac{\epsilon_{\text{km}}}{6371.0088}$$


In [ ]:
clustered_df, metrics, summary_df = run_dbscan_clustering(
    featured_df,
    eps_km=0.45,
    min_samples=15
)
print("DBSCAN Clustering Scorecard:")
for k, v in metrics.items():
    print(f" - {k}: {v}")

print("\nDetected Hotspots Summary Table:")
summary_df


## 3. Methodological Comparison: K-Means vs. DBSCAN


In [ ]:
km_df, km_metrics = run_kmeans_comparison(featured_df, n_clusters=5)
print(f"K-Means Inertia: {km_metrics['inertia']}")
print(f"K-Means Silhouette Score: {km_metrics['silhouette_score']}")
print("\nDiscussion: Why DBSCAN is superior for spatial incident analysis:")
print("1. DBSCAN detects non-spherical clusters of arbitrary geometry.")
print("2. DBSCAN isolates sparse noise instead of forcing outliers into clusters.")
